In [67]:
import xarray as xarray
import rioxarray
import rasterio
import os
import numpy as np
import pandas as pd

In [26]:
year = 2024

In [77]:
fires = pd.read_csv('/home/sgirtsou/Projects/rs_tools/data/fire-detection/nasa_af_corrected_filtered_2024.csv')
fires

,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,INSTRUMENT,VERSION,BRIGHT_T31,FRP,DAYNIGHT,TYPE,layer,path,id,initialdat,area_ha,days_diff
0,36.54770,22.97330,328.00,1.00,1.00,2024/06/26,834,Terra,MODIS,61.03,306.80,14.40,D,0.0,fire_archive_M-C61_579819,/home/sg/Desktop/validated_af/2024/DL_FIRE_M-C...,232024.0,2024/06/26,100.0,0.0
1,36.55128,22.97509,337.82,0.39,0.44,2024/06/26,1151,N,VIIRS,2,310.37,3.62,D,0.0,fire_archive_SV-C2_579822,/home/sg/Desktop/validated_af/2024/DL_FIRE_SV-...,232024.0,2024/06/26,100.0,0.0
2,36.55531,22.97411,330.27,0.39,0.44,2024/06/26,1151,N,VIIRS,2,304.56,2.00,D,0.0,fire_archive_SV-C2_579822,/home/sg/Desktop/validated_af/2024/DL_FIRE_SV-...,232024.0,2024/06/26,100.0,0.0
3,36.55412,22.97298,346.11,0.38,0.36,2024/06/26,1125,N21,VIIRS,2.0NRT,311.99,12.81,D,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,232024.0,2024/06/26,100.0,0.0
4,37.60360,-3.35210,337.90,1.40,1.20,2024/07/24,1419,Aqua,MODIS,61.03,323.30,19.70,D,0.0,fire_archive_M-C61_579819,/home/sg/Desktop/validated_af/2024/DL_FIRE_M-C...,235118.0,2024/07/24,100.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83704,34.03535,36.05851,336.93,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,291.23,9.44,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83705,34.03664,36.04720,302.66,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,288.77,11.00,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83706,34.03215,36.05220,319.44,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,291.39,4.30,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83707,34.03150,36.05790,303.37,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,290.34,1.67,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN


In [78]:
fires = fires.drop_duplicates(subset=["LATITUDE", "LONGITUDE", "ACQ_DATE", "ACQ_TIME", "SATELLITE"])
fires

,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,INSTRUMENT,VERSION,BRIGHT_T31,FRP,DAYNIGHT,TYPE,layer,path,id,initialdat,area_ha,days_diff
0,36.54770,22.97330,328.00,1.00,1.00,2024/06/26,834,Terra,MODIS,61.03,306.80,14.40,D,0.0,fire_archive_M-C61_579819,/home/sg/Desktop/validated_af/2024/DL_FIRE_M-C...,232024.0,2024/06/26,100.0,0.0
1,36.55128,22.97509,337.82,0.39,0.44,2024/06/26,1151,N,VIIRS,2,310.37,3.62,D,0.0,fire_archive_SV-C2_579822,/home/sg/Desktop/validated_af/2024/DL_FIRE_SV-...,232024.0,2024/06/26,100.0,0.0
2,36.55531,22.97411,330.27,0.39,0.44,2024/06/26,1151,N,VIIRS,2,304.56,2.00,D,0.0,fire_archive_SV-C2_579822,/home/sg/Desktop/validated_af/2024/DL_FIRE_SV-...,232024.0,2024/06/26,100.0,0.0
3,36.55412,22.97298,346.11,0.38,0.36,2024/06/26,1125,N21,VIIRS,2.0NRT,311.99,12.81,D,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,232024.0,2024/06/26,100.0,0.0
4,37.60360,-3.35210,337.90,1.40,1.20,2024/07/24,1419,Aqua,MODIS,61.03,323.30,19.70,D,0.0,fire_archive_M-C61_579819,/home/sg/Desktop/validated_af/2024/DL_FIRE_M-C...,235118.0,2024/07/24,100.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83704,34.03535,36.05851,336.93,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,291.23,9.44,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83705,34.03664,36.04720,302.66,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,288.77,11.00,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83706,34.03215,36.05220,319.44,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,291.39,4.30,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN
83707,34.03150,36.05790,303.37,0.54,0.42,2024/09/27,2303,N21,VIIRS,2.0NRT,290.34,1.67,N,NaN,fire_nrt_J2V-C2_579821,/home/sg/Desktop/validated_af/2024/DL_FIRE_J2V...,NaN,NaN,NaN,NaN


In [79]:
fires.columns

Index(['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'SCAN', 'TRACK', 'ACQ_DATE',
       'ACQ_TIME', 'SATELLITE', 'INSTRUMENT', 'VERSION', 'BRIGHT_T31', 'FRP',
       'DAYNIGHT', 'TYPE', 'layer', 'path', 'id', 'initialdat', 'area_ha',
       'days_diff'],
      dtype='object')

In [80]:
fires_grouped = fires.groupby(["ACQ_DATE", "ACQ_TIME", "SATELLITE", "id"])["FRP"].count().reset_index().sort_values(by=["FRP"], ascending=True)

In [81]:
to_delete = fires_grouped[fires_grouped.FRP < 10].reset_index()
to_delete.shape

(7249, 6)

In [82]:
fires.shape

(80830, 20)

In [83]:
for index, row in to_delete.iterrows():
    fires = fires[~(
        (fires["ACQ_DATE"] == row["ACQ_DATE"]) &
        (fires["ACQ_TIME"] == row["ACQ_TIME"]) &
        (fires["SATELLITE"] == row["SATELLITE"]) &
        (fires["id"] == row["id"])
    )]

In [84]:
fires.shape

(57705, 20)

In [85]:
fires.to_csv('/home/sgirtsou/Projects/rs_tools/data/fire-detection/nasa_af_point_cloud_10_2024.csv')